<a href="https://colab.research.google.com/github/RashaHE/Comparative-Oncology-ORA-Engin/blob/main/Compare_Eng.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Comparative Engin Function**:
This function is designed to apply 2 main objectives in comparative oncology anlysis.
**1st:** Compare 2 ORA objects (even if of different species) to detect common pathways across these objects;
**2nd:** automatically slice open the data to extract exact cross-species genes that drive these pathways.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:

drive_Path<-"/content/drive/MyDrive/Colab Notebooks/R.packages"
.libPaths(c(drive_Path,.libPaths()))

In [2]:
if(!require("BiocManager", quietly=TRUE)){install.packages("BiocManager",lib = drive_Path)}

In [3]:
library(dplyr)
library(ggplot2)
library(tidyverse)
if (!requireNamespace("ggrepel", quietly = TRUE)) {install.packages("ggrepel", lib= drive_Path)}
library(ggrepel)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ lubridate 1.9.5     ✔ tibble    3.3.1
✔ purrr     1.2.2     ✔ tidyr     1.3.2
✔ readr     2.2.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [4]:
BiocManager::install(c("clusterProfiler",
  "org.Hs.eg.db",
  "org.Cf.eg.db",
  "biomaRt",
  "enrichplot"
,lib=drive_Path))
install.packages("ggVennDiagram",lib = drive_Path)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)

Warning message:
“package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'clusterProfiler' 'org.Hs.eg.db' 'org.Cf.eg.db'
  'biomaRt' 'enrichplot'”
Warning message:
“packages not installed (unknown repository) ''/content/drive/MyDrive/Colab
  Notebooks/R.packages''”
Old packages: 'limma', 'bit64', 'xtable'



In [5]:
compar_Eng<-function(ora_obj1, ora_obj2, name1= "Dataset1",name2= "Dataset2"){
  df1<-as.data.frame(ora_obj1)
  df2<-as.data.frame(ora_obj2)
  common_paths<-intersect(df1$Description,df2$Description)
  if(length(common_paths)==0){
    print("No_shared_pathways")
  return(NULL)
    }
  master_list<-list()
    for(i in common_paths){
      df1_genes<-strsplit(df1$geneID[df1$Description==i],"/")[[1]] ####change the strsplit separator from a comma "," to a slash "/" depending on the used dataset).
      df2_genes<-strsplit(df2$geneID[df2$Description==i],"/")[[1]]
   master_list[[i]]<-list(df1_genes,df2_genes)
   names(master_list[[i]])<-c (paste0(name1,"_genes"),paste0(name2,"_genes"))

      }
return(master_list)
}

In [6]:
library(clusterProfiler)
library(org.Hs.eg.db)
library(DOSE)



clusterProfiler v4.20.0 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

G Yu. Thirteen years of clusterProfiler. The Innovation. 2024,
5(6):100722


Attaching package: ‘clusterProfiler’


The following object is masked from ‘package:purrr’:

    simplify


The following object is masked from ‘package:stats’:

    filter


Loading required package: AnnotationDbi

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following object is masked from ‘package:lubridate’:

    as.difftime


The following object is masked from ‘package:dplyr’:

    explain


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following object is masked from ‘package:dplyr’:

    combine


The following objects are masked from ‘package:stats’:

  

In [7]:
data(geneList, package="DOSE")

Example Excercise

In [8]:
fake_human_genes <- names(geneList)[1:200]


In [9]:
fake_canine_genes <- names(geneList)[100:300]

In [10]:
print("Running ORA 1... ")
ora_human <- enrichGO(gene          = fake_human_genes,
                      OrgDb         = org.Hs.eg.db,
                      ont           = "BP",
                      pAdjustMethod = "fdr",
                      pvalueCutoff  = 0.05)

print("Running ORA 2... ")
ora_canine <- enrichGO(gene          = fake_canine_genes,
                       OrgDb         = org.Hs.eg.db,
                       ont           = "BP",
                       pAdjustMethod = "fdr",
                       pvalueCutoff  = 0.05)

print("Success! You now have two real ORA objects in your environment.")

[1] "Running ORA 1... "
[1] "Running ORA 2... "
[1] "Success! You now have two real ORA objects in your environment."


In [11]:
Results<-compar_Eng(ora_human,ora_canine,name1 = "Human",name2 = "Canine")

In [12]:
print(names(Results))

 [1] "nuclear chromosome segregation"                                         
 [2] "chromosome segregation"                                                 
 [3] "sister chromatid segregation"                                           
 [4] "nuclear division"                                                       
 [5] "mitotic sister chromatid segregation"                                   
 [6] "mitotic nuclear division"                                               
 [7] "organelle fission"                                                      
 [8] "regulation of chromosome segregation"                                   
 [9] "positive regulation of cell cycle process"                              
[10] "positive regulation of cell cycle"                                      
[11] "mitotic cell cycle phase transition"                                    
[12] "regulation of chromosome separation"                                    
[13] "regulation of sister chromatid segregation"   